## 🌍 CASE 3: Flight Route & Schedule Optimization
**Objective:**
To analyze and optimize global airline routes by identifying the most connected airports, popular flight paths, and shortest travel routes using graph-based analytics in Spark.

### Step 1: Load & Inspect Data

In [0]:
df = spark.table("workspace.default.3_flight_route_database")
# Inspect schema and sample
df.printSchema()
display(df.limit(5))


root
 |-- airline: string (nullable = true)
 |-- airline ID: string (nullable = true)
 |--  source airport: string (nullable = true)
 |--  source airport id: string (nullable = true)
 |--  destination apirport: string (nullable = true)
 |--  destination airport id: string (nullable = true)
 |--  codeshare: string (nullable = true)
 |--  stops: long (nullable = true)
 |--  equipment: string (nullable = true)



airline,airline ID,source airport,source airport id,destination apirport,destination airport id,codeshare,stops,equipment
2B,410,AER,2965,KZN,2990,null,0,CR2
2B,410,ASF,2966,KZN,2990,null,0,CR2
2B,410,ASF,2966,MRV,2962,null,0,CR2
2B,410,CEK,2968,KZN,2990,null,0,CR2
2B,410,CEK,2968,OVB,4078,null,0,CR2


### Step 2: Clean & Standardize Column Names

In [0]:
import re
from pyspark.sql.functions import trim, col

def clean_column_name(name):
    return re.sub(r'[^A-Za-z0-9]+', '_', name.strip())

df = df.toDF(*[clean_column_name(c) for c in df.columns])
df = df.select([trim(col(c)).alias(c) for c in df.columns])

print("✅ Cleaned column names:")
print(df.columns)
display(df.limit(5))


✅ Cleaned column names:
['airline', 'airline_ID', 'source_airport', 'source_airport_id', 'destination_apirport', 'destination_airport_id', 'codeshare', 'stops', 'equipment']


airline,airline_ID,source_airport,source_airport_id,destination_apirport,destination_airport_id,codeshare,stops,equipment
2B,410,AER,2965,KZN,2990,null,0,CR2
2B,410,ASF,2966,KZN,2990,null,0,CR2
2B,410,ASF,2966,MRV,2962,null,0,CR2
2B,410,CEK,2968,KZN,2990,null,0,CR2
2B,410,CEK,2968,OVB,4078,null,0,CR2


### Step 3: Data Cleaning & Filtering
Removing any null or incomplete route records.

In [0]:
df = df.filter(
    (col("source_airport").isNotNull()) &
    (col("destination_apirport").isNotNull()) &
    (col("stops") == "0")  # 👈 string, because free edition treats all CSV as string
)

print("✅ Cleaned routes count:", df.count())
display(df.limit(5))


✅ Cleaned routes count: 67652


airline,airline_ID,source_airport,source_airport_id,destination_apirport,destination_airport_id,codeshare,stops,equipment
2B,410,AER,2965,KZN,2990,null,0,CR2
2B,410,ASF,2966,KZN,2990,null,0,CR2
2B,410,ASF,2966,MRV,2962,null,0,CR2
2B,410,CEK,2968,KZN,2990,null,0,CR2
2B,410,CEK,2968,OVB,4078,null,0,CR2


### Step 4: SQL Exploratory Data Analysis (EDA)
Perform SQL-based exploration to find:
- Airports with most outgoing routes.
- Airlines operating the highest number of routes.

In [0]:
df.createOrReplaceTempView("routes")

# Top 10 most connected source airports
spark.sql("""
SELECT source_airport, COUNT(*) AS total_routes
FROM routes
GROUP BY source_airport
ORDER BY total_routes DESC
LIMIT 10
""").show()

# Top airlines by total routes
spark.sql("""
SELECT Airline, COUNT(*) AS route_count
FROM routes
GROUP BY Airline
ORDER BY route_count DESC
LIMIT 10
""").show()


+--------------+------------+
|source_airport|total_routes|
+--------------+------------+
|           ATL|         915|
|           ORD|         558|
|           PEK|         535|
|           LHR|         527|
|           CDG|         524|
|           FRA|         497|
|           LAX|         492|
|           DFW|         469|
|           JFK|         456|
|           AMS|         453|
+--------------+------------+

+-------+-----------+
|Airline|route_count|
+-------+-----------+
|     FR|       2484|
|     AA|       2354|
|     UA|       2180|
|     DL|       1981|
|     US|       1960|
|     CZ|       1454|
|     MU|       1263|
|     CA|       1260|
|     WN|       1143|
|     U2|       1130|
+-------+-----------+



### Step 5: Graph Construction using NetworkX
This enables graph-based analytics like degree centrality, PageRank, and shortest paths.

In [0]:
# Install networkx
%pip install networkx

# Convert Spark DF → Pandas (since dataset is small enough)
routes_pd = df.select("source_airport", "destination_apirport", "Airline").toPandas()

# Create directed graph
G = nx.from_pandas_edgelist(routes_pd, "source_airport", "destination_apirport", create_using=nx.DiGraph())

print("✅ Graph created with", len(G.nodes), "airports and", len(G.edges), "routes.")


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Graph created with 3425 airports and 37595 routes.


### Step 6: Compute Degree Centrality (Connectivity)
Measure how connected each airport is (number of direct routes).
Higher degree = more connected hub.

In [0]:
deg_data = nx.degree_centrality(G)
deg_df = spark.createDataFrame([(k, float(v)) for k, v in deg_data.items()], ["Airport", "Connectivity"])
deg_df.createOrReplaceTempView("airport_connectivity")

print("✅ Airport connectivity calculated")
spark.sql("""
SELECT Airport, ROUND(Connectivity,4) AS Connectivity
FROM airport_connectivity
ORDER BY Connectivity DESC
LIMIT 10
""").show()


✅ Airport connectivity calculated
+-------+------------+
|Airport|Connectivity|
+-------+------------+
|    FRA|      0.1393|
|    CDG|      0.1373|
|    AMS|      0.1352|
|    IST|      0.1335|
|    ATL|      0.1265|
|    PEK|      0.1203|
|    ORD|      0.1195|
|    MUC|       0.111|
|    DME|      0.1104|
|    DFW|      0.1086|
+-------+------------+



### Step 7: Compute PageRank (Importance)
Find globally important airports using PageRank — identifies major transit hubs like LHR, DXB, ATL, etc.

In [0]:
pr_data = nx.pagerank(G)
pr_df = spark.createDataFrame([(k, float(v)) for k, v in pr_data.items()], ["Airport", "PageRank"])
pr_df.createOrReplaceTempView("airport_pagerank")

print("✅ Airport PageRank calculated")
spark.sql("""
SELECT Airport, ROUND(PageRank,5) AS Importance
FROM airport_pagerank
ORDER BY Importance DESC
LIMIT 10
""").show()


✅ Airport PageRank calculated
+-------+----------+
|Airport|Importance|
+-------+----------+
|    ATL|   0.00467|
|    IST|   0.00439|
|    ORD|   0.00428|
|    DEN|   0.00425|
|    DFW|   0.00418|
|    DME|   0.00411|
|    CDG|   0.00394|
|    FRA|   0.00383|
|    PEK|   0.00381|
|    DXB|   0.00364|
+-------+----------+



### Step 8: Find Shortest Route

In [0]:
try:
    path = nx.shortest_path(G, source="DEL", target="JFK")
    print("Shortest path from DEL to JFK:", " → ".join(path))
except Exception as e:
    print("⚠️ No direct route found between DEL and JFK or not in dataset.")


Shortest path from DEL to JFK: DEL → JFK


### Step 9: SQL Visualization

In [0]:
%sql
-- Top airports by connectivity
SELECT Airport, ROUND(Connectivity,4) AS Connectivity
FROM airport_connectivity
ORDER BY Connectivity DESC
LIMIT 10;

-- Top airports by PageRank
SELECT Airport, ROUND(PageRank,5) AS Importance
FROM airport_pagerank
ORDER BY Importance DESC
LIMIT 10;


Airport,Importance
ATL,0.00467
IST,0.00439
ORD,0.00428
DEN,0.00425
DFW,0.00418
DME,0.00411
CDG,0.00394
FRA,0.00383
PEK,0.00381
DXB,0.00364


Databricks visualization. Run in Databricks to view.